# Session 2 — Pandas Deeper: Real Data is Messy

**Module:** M3 — Data Foundation  
**Prerequisites:** Session 1 (DataFrames, filtering, groupby, Streamlit intro)  
**Duration:** ~3 hours live + homework  

---

## What we are building toward

By Session 15 we will ship a multi-agent AI system that investigates **real ZF Life manufacturing data**.
That data will not be clean. It will never be clean. Every professional dataset you will ever encounter has:

- missing values in the worst possible columns
- dates stored in five different formats inside the same column
- numeric fields that somehow contain the string `"err"`
- duplicate rows because sensors fire twice
- categories spelled ten different ways (`"Morning"`, `"morning"`, `"AM"`, `"mornng"`)

If you can only handle tidy data, you are stuck the moment you touch the real project.
This session fixes that.

---

## Session objectives

1. Understand **why Parquet** instead of CSV for multi-table, time series data
2. Diagnose missing values precisely and choose a strategy for each column
3. Coerce mixed-type columns back to the right dtype without losing data
4. Detect and remove duplicates safely
5. Normalise inconsistent category labels
6. Merge three manufacturing tables on shared keys
7. Work with datetime columns: extract week, shift block, rolling window
8. Chain all of these operations cleanly into a reusable pipeline

---

## Part 0 — Why Parquet? (and why we are leaving CSV behind)

In Session 1 we loaded a CSV file. CSV is simple and universal, which makes it a great teaching tool.
But as soon as your data grows beyond a few thousand rows or involves multiple tables, CSV starts showing its problems.

### The problem with CSV

| Problem | What happens |
|---|---|
| No type information | Every column is loaded as a string. Pandas guesses (and sometimes guesses wrong). |
| No compression | A 100 MB CSV stays 100 MB on disk. |
| Row-by-row parsing | Reading 1 million rows means scanning every character. It is slow. |
| No column pruning | If you only need 3 of 50 columns, you still pay to load all 50. |
| No schema enforcement | Anyone can add or remove a column between writes and you will not know until the code crashes. |

### Why Parquet solves all of this

**Parquet** is a binary, columnar file format created by Apache.
`pyarrow` (the library that powers `pandas.to_parquet()`) is the standard Python interface to it.

| Feature | How Parquet handles it |
|---|---|
| **Schema** | Types are stored in the file header. `int64` stays `int64`. Dates stay dates. |
| **Compression** | Built-in Snappy or GZIP compression. Typically 3–10x smaller than CSV. |
| **Columnar storage** | Reading only column `machine_id` scans only that column's bytes, not the whole file. |
| **Predicate pushdown** | With tools like DuckDB (Session 3), you can filter at the storage layer before loading into memory. |
| **Row groups** | Data is chunked into row groups. Parallel reads become trivial. |

### When to still use CSV

CSV is fine when:
- you need a human to open the file in Excel or a text editor
- you are exchanging data with a system that does not support Parquet
- the file is under ~5 000 rows and only lives for one analysis

For everything else — especially the ZF Life project — use Parquet.

> **Mentor note:** Think of CSV as a plain text letter and Parquet as a compressed, indexed database file.
> Both carry information, but one is built for machines to read at speed.

---

## Part 1 — Generating the messy datasets

We have a Python script called `generate_parquet_data.py` in this folder.
It creates three Parquet files inside `./data/`:

| File | Description | ~Rows |
|---|---|---|
| `production_log.parquet` | One row per production cycle. Main fact table. | 12 240 |
| `inspection_log.parquet` | One row per quality inspection event. | 7 956 |
| `material_batches.parquet` | One row per material batch delivered. | 505 |

All three files are intentionally broken in realistic ways.
Our job this session is to diagnose and fix them.

Run the cell below to generate the files.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "generate_parquet_data.py", "--rows", "12000", "--seed", "42"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

---

## Part 2 — Loading Parquet files and first inspection

Loading a Parquet file in Pandas is one function call, just like CSV.
The key difference: types are preserved automatically.

```python
df = pd.read_parquet("data/production_log.parquet")
```

We will load all three tables and then start our investigation.

In [ ]:
import pandas as pd
import numpy as np

prod = pd.read_parquet("data/production_log.parquet")
insp = pd.read_parquet("data/inspection_log.parquet")
mats = pd.read_parquet("data/material_batches.parquet")

print(f"production_log  : {prod.shape[0]:>6,} rows * {prod.shape[1]} columns")
print(f"inspection_log  : {insp.shape[0]:>6,} rows * {insp.shape[1]} columns")
print(f"material_batches: {mats.shape[0]:>6,} rows * {mats.shape[1]} columns")

production_log  : 12,240 rows * 12 columns
inspection_log  :  7,956 rows * 11 columns
material_batches:    505 rows * 6 columns


### First look — production_log

Before cleaning anything, always look at the raw data first.
`df.head()` shows the first five rows.
`df.info()` shows column names, non-null counts, and inferred dtypes.

Pay attention to columns where the dtype is `object` but you expected a number or a date.
That is almost always a sign that something went wrong during data collection.

In [3]:
prod.head(10)

,batch_id,timestamp,machine_id,shift,operator_id,product_code,units_produced,defective_units,cycle_time_sec,temperature_c,pressure_bar,material_batch_id
0,B09786,2027-12-17 23:56:22,M06,Eve,OP014,PC-1100,454.0,53.0,51.61,85.34,9.241,MB0087
1,B05507,2026-08-30 05:39:46,M01,AM,OP019,PC-1100,421.0,45.0,53.29,81.70,8.585,MB0416
2,B02272,2025-09-08T17:32:35,M01,Morning,OP006,PC-2100,397.0,30.0,50.72,76.53,9.989,MB0276
3,B03885,2026-03-04 05:04:21,M06,Morning,OP002,PC-3300,387.0,25.0,29.24,66.20,10.687,NaN
4,B08581,2027-08-09T05:53:29,M05,morning,OP009,PC-2100,345.0,38.0,45.35,72.64,8.841,MB0251
5,B10310,2028-02-14 02:42:49,M03,Evening,OP018,PC-2100,539.0,50.0,47.5,72.39,9.762,MB0130
6,B04639,2026-05-29 04:29:13,M03,N,OP020,PC-1100,581.0,41.0,31.91,77.07,7.850,MB0031
7,B06269,2026-11-23 20:27:19,M06,Night,OP003,PC-4400,517.0,51.0,50.46,68.58,6.423,NaN
8,B11274,2028-06-03 05:49:14,M06,EVENING,OP012,PC-4400,481.0,24.0,51.19,77.61,6.792,MB0005
9,B06396,2026-12-05 20:18:43,M06,PM,OP017,PC-1200,610.0,72.0,35.99,75.97,7.714,MB0340


In [4]:
prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 12240 entries, 0 to 12239
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   batch_id           12240 non-null  str    
 1   timestamp          12240 non-null  str    
 2   machine_id         12240 non-null  str    
 3   shift              12240 non-null  str    
 4   operator_id        11628 non-null  str    
 5   product_code       12240 non-null  str    
 6   units_produced     12240 non-null  str    
 7   defective_units    12240 non-null  str    
 8   cycle_time_sec     12240 non-null  str    
 9   temperature_c      11620 non-null  float64
 10  pressure_bar       11614 non-null  float64
 11  material_batch_id  11645 non-null  str    
dtypes: float64(2), str(10)
memory usage: 1.9 MB


---

## Part 3 — Missing Values: diagnose before you fix

The single most common mistake when cleaning data is filling in missing values without understanding *why* they are missing.

There are three types of missingness:

| Type | Meaning | Example |
|---|---|---|
| **MCAR** (Missing Completely At Random) | The gap is random and unrelated to the data | A sensor dropped a packet for 2 seconds |
| **MAR** (Missing At Random) | The gap depends on another observed variable | Night-shift operators skip the notes field |
| **MNAR** (Missing Not At Random) | The gap is caused by the value itself | Very high temperatures cause the sensor to fail |

MNAR is the dangerous one. If you fill it in with a mean or zero, you are lying about the data.

**Rule of thumb:**
- Look at missing value *patterns*, not just counts
- Ask: "Is the gap random or does it happen for specific machines/shifts/operators?"
- Document every decision you make

In [5]:
# Step 1: Count missing values per column and compute the percentage
missing_summary = pd.DataFrame({
    "missing_count": prod.isna().sum(),
    "missing_pct": (prod.isna().mean() * 100).round(2),
    "dtype": prod.dtypes,
}).sort_values("missing_pct", ascending=False)

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_pct,dtype
pressure_bar,626,5.11,float64
temperature_c,620,5.07,float64
operator_id,612,5.00,str
material_batch_id,595,4.86,str


### Strategy decisions

For each column with missing values, we need a deliberate strategy.
The code below documents and implements each one.

| Column | Strategy | Reason |
|---|---|---|
| `temperature_c` | Fill with rolling median per machine | Sensor dropout — nearby readings are the best estimate |
| `pressure_bar` | Fill with rolling median per machine | Same as above |
| `operator_id` | Fill with `"UNKNOWN"` | We cannot guess the operator; flag it instead |
| `material_batch_id` | Fill with `"MISSING"` | Important for join integrity — flag it |

> **Instructor note:** Ask students — *why not fill `temperature_c` with the global mean?*
> Because machine M01 might run at 65°C and M05 at 80°C. A global mean would introduce noise, not reduce it.

In [6]:
# Work on a copy so the original stays untouched for comparison
prod_clean = prod.copy()

# Categorical columns — fill with explicit sentinel values
prod_clean["operator_id"] = prod_clean["operator_id"].fillna("UNKNOWN")
prod_clean["material_batch_id"] = prod_clean["material_batch_id"].fillna("MISSING")

# Numeric columns — fill with median per machine group
for col in ["temperature_c", "pressure_bar"]:
    group_medians = prod_clean.groupby("machine_id")[col].transform("median")
    prod_clean[col] = prod_clean[col].fillna(group_medians)

# Verify: no more missing values in the targeted columns
still_missing = prod_clean[["operator_id", "material_batch_id", "temperature_c", "pressure_bar"]].isna().sum()
print("Remaining missing values after fill:")
print(still_missing)

Remaining missing values after fill:
operator_id          0
material_batch_id    0
temperature_c        0
pressure_bar         0
dtype: int64


---

## Part 4 — Data Types: the silent killers

In Session 1 we saw that Pandas reads CSV columns and guesses the type.
With Parquet, types are stored — but our generator deliberately introduced corruption:

- `units_produced` and `defective_units` contain values like `"N/A"` or `"err"` — so Pandas stored the whole column as `object` (string)
- `timestamp` contains dates in mixed string formats

Both problems are extremely common in real manufacturing exports from SCADA systems and ERPs.

The tool we use for both is:

```python
pd.to_numeric(series, errors='coerce')   # bad values → NaN
pd.to_datetime(series, errors='coerce')  # bad dates → NaT
```

`errors='coerce'` is the key. Instead of crashing on bad values, it turns them into `NaN` / `NaT`.
This lets us convert the whole column first, then handle the new nulls separately.

In [7]:
# Inspect what the corrupted numeric columns actually contain
print("Sample of units_produced dtype:", prod_clean["units_produced"].dtype)
print()

# Show rows where units_produced is NOT a valid number
bad_mask = pd.to_numeric(prod_clean["units_produced"], errors="coerce").isna()
bad_rows = prod_clean.loc[bad_mask, ["batch_id", "machine_id", "units_produced", "defective_units"]]
print(f"Rows with non-numeric units_produced: {bad_mask.sum()}")
bad_rows.head(8)

Sample of units_produced dtype: str

Rows with non-numeric units_produced: 234


,batch_id,machine_id,units_produced,defective_units
65,B08410,M03,#VALUE!,39.0
202,B06874,M06,null,24.0
219,B08609,M05,N/A,15.0
223,B07220,M04,err,12.0
239,B06087,M01,N/A,12.0
267,B07652,M03,#VALUE!,21.0
310,B07861,M01,err,38.0
378,B01211,M04,null,12.0


In [9]:
# Fix: coerce numeric columns — bad values become NaN, then fill with column median
for col in ["units_produced", "defective_units", "cycle_time_sec"]:
    prod_clean[col] = pd.to_numeric(prod_clean[col], errors="coerce")
    median_val = prod_clean[col].median()
    prod_clean[col] = prod_clean[col].fillna(median_val)

# Fix: parse the timestamp column — mixed format strings become proper datetime objects
# pandas automatically infers the format; errors="coerce" turns unparseable values into NaT
prod_clean["timestamp"] = pd.to_datetime(prod_clean["timestamp"], errors="coerce")

# How many timestamps failed to parse?
n_nat = prod_clean["timestamp"].isna().sum()
print(f"Timestamps that could not be parsed (NaT): {n_nat}")

# Verify the final dtypes
prod_clean[["units_produced", "defective_units", "cycle_time_sec", "timestamp"]].dtypes

Timestamps that could not be parsed (NaT): 487


units_produced            float64
defective_units           float64
cycle_time_sec            float64
timestamp          datetime64[us]
dtype: object

### Removing physical outliers

Our generator also injected values like `units_produced = 87 000`.
A machine that produces 300–650 units per cycle cannot produce 87 000 in the same cycle.
These are **physically impossible values** — not extreme-but-valid, but straight-up wrong.

We cap them using domain knowledge: max realistic output for this factory is ~800 units per cycle.

In [10]:
# Flag physically impossible outliers before removing them
outlier_mask = prod_clean["units_produced"] > 800
print(f"Outlier rows (units_produced > 800): {outlier_mask.sum()}")
print(prod_clean.loc[outlier_mask, ["batch_id", "machine_id", "units_produced"]].head())

# Cap at domain-knowledge maximum instead of dropping (keeps the row, fixes the value)
prod_clean["units_produced"] = prod_clean["units_produced"].clip(upper=800)

Outlier rows (units_produced > 800): 67
    batch_id machine_id  units_produced
68    B07005        M01          9470.0
354   B08455        M05         98926.0
492   B10323        M04         31338.0
507   B05337        M01         69038.0
695   B06391        M03         92168.0


In [15]:
# Count how many records exist for each units_produced value
grouped = prod_clean.groupby("units_produced")["units_produced"]
grouped.count()

units_produced
300.0    33
301.0    43
302.0    39
303.0    33
304.0    31
         ..
646.0    24
647.0    26
648.0    32
649.0    27
800.0    67
Name: units_produced, Length: 351, dtype: int64

---

## Part 5 — Duplicates: quieter than missing values, equally dangerous

Duplicate rows inflate counts, distort averages, and make trend lines spike.
In manufacturing, they typically come from:

- a sensor that fires twice on the same event
- a database export that ran twice
- a manual copy-paste

The Pandas workflow is:
1. `df.duplicated()` — returns a boolean mask of duplicate rows
2. `df.drop_duplicates()` — removes them

**Important:** decide *which columns define uniqueness* before dropping.
For a production log, `batch_id` should be unique. Two rows with the same `batch_id` are always a duplicate, even if other columns differ.

In [16]:
# Check for fully identical rows first
full_dupes = prod_clean.duplicated().sum()
print(f"Fully identical duplicate rows: {full_dupes}")

# Then check for duplicate batch_ids (semantically duplicated even if other cols differ)
batch_dupes = prod_clean.duplicated(subset=["batch_id"]).sum()
print(f"Rows with a duplicate batch_id: {batch_dupes}")

# Show a sample of the duplicated batch_ids
dupe_ids = prod_clean.loc[prod_clean.duplicated(subset=["batch_id"], keep=False), "batch_id"].unique()[:5]
prod_clean[prod_clean["batch_id"].isin(dupe_ids)].sort_values("batch_id").head(10)

Fully identical duplicate rows: 240
Rows with a duplicate batch_id: 240


,batch_id,timestamp,machine_id,shift,operator_id,product_code,units_produced,defective_units,cycle_time_sec,temperature_c,pressure_bar,material_batch_id
18,B04917,2026-07-01 23:33:40,M02,Morning,OP010,PC-2100,523.0,27.0,38.18,61.21,8.287,MB0382
8873,B04917,2026-07-01 23:33:40,M02,Morning,OP010,PC-2100,523.0,27.0,38.18,61.21,8.287,MB0382
81,B06166,2026-11-12 05:38:44,M01,EVENING,OP011,PC-3300,502.0,20.0,46.36,71.92,10.176,MB0386
7596,B06166,2026-11-12 05:38:44,M01,EVENING,OP011,PC-3300,502.0,20.0,46.36,71.92,10.176,MB0386
31,B07019,2027-02-12 10:45:35,M04,Morning,OP014,PC-3300,457.0,5.0,51.52,75.12,7.905,MB0067
8280,B07019,2027-02-12 10:45:35,M04,Morning,OP014,PC-3300,457.0,5.0,51.52,75.12,7.905,MB0067
76,B09027,2027-09-26 16:27:50,M05,Morning,OP009,PC-3300,549.0,21.0,54.83,72.46,7.925,MB0430
10915,B09027,2027-09-26 16:27:50,M05,Morning,OP009,PC-3300,549.0,21.0,54.83,72.46,7.925,MB0430
97,B09808,2027-12-20 22:15:30,M04,Eve,OP016,PC-3300,588.0,65.0,36.29,80.35,7.226,MB0239
11035,B09808,2027-12-20 22:15:30,M04,Eve,OP016,PC-3300,588.0,65.0,36.29,80.35,7.226,MB0239


In [17]:
rows_before = len(prod_clean)

# keep='first' → keep the first occurrence, drop all later duplicates
prod_clean = prod_clean.drop_duplicates(subset=["batch_id"], keep="first").reset_index(drop=True)

rows_after = len(prod_clean)
print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Removed:     {rows_before - rows_after:,} duplicate rows")

Rows before: 12,240
Rows after:  12,000
Removed:     240 duplicate rows


---

## Part 6 — Inconsistent Categories: normalising the chaos

Look at the `shift` column. It should have three values: `Morning`, `Evening`, `Night`.
What it actually contains is a long tail of creative spellings.

This is one of the most common real-world data quality problems.
Different operators, different data entry systems, and different export scripts
all produce slightly different strings for the same concept.

Our approach:
1. `value_counts()` to see all variants and their frequency
2. Build a mapping dictionary: every known variant → canonical value
3. Apply with `.map()` or `.replace()`
4. Mark any unmapped value as `"UNKNOWN"` so we can see what the map missed

In [21]:
# See all shift variants and their frequency
print("All shift values in the dataset:")
print(prod_clean["shift"].value_counts(dropna=False))

All shift values in the dataset:
shift
Night      1257
N          1247
morning    1229
AM         1212
night      1194
Eve        1187
PM         1187
EVENING    1174
Evening    1157
Morning    1156
Name: count, dtype: int64


In [22]:
SHIFT_MAP = {
    "Morning": "Morning", "morning": "Morning", "AM":      "Morning",
    "Evening": "Evening", "EVENING": "Evening", "Eve":     "Evening",
    "Night":   "Night",   "night":   "Night",   "PM":      "Night", "N": "Night",
}

prod_clean["shift_clean"] = prod_clean["shift"].map(SHIFT_MAP).fillna("UNKNOWN")

# Verify: after mapping, only canonical values + UNKNOWN should exist
print("Values after normalisation:")
print(prod_clean["shift_clean"].value_counts(dropna=False).to_string())

Values after normalisation:
shift_clean
Night      4885
Morning    3597
Evening    3518


> **Challenge for students:**  
> The `defect_type` column in the `inspection_log` table has the same problem — about 18 different spellings for 6 categories.  
> Can you write the mapping dictionary and clean that column without looking at the answer?
> 
> Hint: start with `insp["defect_type"].value_counts()`.

---

## Part 7 — Merging Tables: connecting the full picture

Manufacturing data lives in multiple tables. A single production record does not carry all the context you need to answer business questions. You need to join:

- production event (what happened, when, on which machine)
- inspection result (was it defective, what type, how severe)
- material batch (which supplier, which grade)

Only by joining all three can you answer questions like:
> *"Do defect rates differ by material supplier?"*
> *"Does Supplier B material produce more weld defects on night shift?"*

### Merge types recap

| Join type | What it keeps |
|---|---|
| `inner` | Only rows that match in both tables |
| `left` | All rows from the left table; NaN where no match on the right |
| `right` | All rows from the right table; NaN where no match on the left |
| `outer` | All rows from both; NaN wherever there is no match |

For production analysis we almost always use `left` join:
keep all production records, enrich with inspection and material data where available.

In [23]:
# First: clean up mats and insp columns we need for joining
mats_clean = mats.drop_duplicates(subset=["material_batch_id"], keep="first").copy()

# Step 1: join production + material batches
df_merged = prod_clean.merge(
    mats_clean[["material_batch_id", "supplier", "material_grade", "certificate_ok"]],
    on="material_batch_id",
    how="left",
)

print(f"After production + material merge: {df_merged.shape}")
df_merged[["batch_id", "machine_id", "supplier", "material_grade", "certificate_ok"]].head()

After production + material merge: (12000, 16)


,batch_id,machine_id,supplier,material_grade,certificate_ok
0,B09786,M06,SupplierA,Grade-A,True
1,B05507,M01,SupplierB,Grade-B,True
2,B02272,M01,SupplierA,Grade-A,True
3,B03885,M06,NaN,NaN,<NA>
4,B08581,M05,SupplierA,grade-a,<NA>


In [26]:
# Step 2: aggregate inspection results to batch level before joining
# (one batch can have multiple inspections — we summarise them)
insp_clean = insp.drop_duplicates(subset=["inspection_id"], keep="first").copy()
insp_clean["units_rejected"] = pd.to_numeric(insp_clean["units_rejected"], errors="coerce")

insp_agg = (
    insp_clean
    .groupby("batch_id")
    .agg(
        n_inspections=("inspection_id", "count"),
        total_units_rejected=("units_rejected", "sum"),
        most_common_defect=("defect_type", lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else None),
        any_critical=("severity", lambda s: "Critical" in s.str.upper().fillna("").values),
    )
    .reset_index()
)

# Step 3: join aggregated inspection data onto the merged table
df_full = df_merged.merge(insp_agg, on="batch_id", how="left")

print(f"Final merged table: {df_full.shape}")
df_full[["batch_id", "machine_id", "supplier", "n_inspections", "total_units_rejected", "any_critical"]].head(8)

Final merged table: (12000, 20)


,batch_id,machine_id,supplier,n_inspections,total_units_rejected,any_critical
0,B09786,M06,SupplierA,2.0,20.0,False
1,B05507,M01,SupplierB,NaN,NaN,NaN
2,B02272,M01,SupplierA,1.0,0.0,False
3,B03885,M06,NaN,1.0,19.0,False
4,B08581,M05,SupplierA,2.0,11.0,False
5,B10310,M03,SupplierB,1.0,10.0,False
6,B04639,M03,SupplierA,2.0,15.0,False
7,B06269,M06,NaN,3.0,13.0,False


---

## Part 8 — Datetime Operations: the time dimension of manufacturing

Manufacturing problems are almost always time-dependent:

- defect rates drift up over a shift as operators get tired
- machine wear accumulates over weeks
- a bad material batch affects a 3-day window before the next delivery
- anomalies often cluster around maintenance events

To see these patterns you need to extract structured time features from your timestamp column
and compute rolling statistics.

All Pandas datetime operations live under `.dt`:

```python
df["timestamp"].dt.hour          # → 0..23
df["timestamp"].dt.day_of_week   # → 0=Monday, 6=Sunday
df["timestamp"].dt.isocalendar().week  # → ISO week number
```

In [27]:
# Drop rows where timestamp failed to parse (NaT)
df_time = df_full.dropna(subset=["timestamp"]).copy()
df_time = df_time.sort_values("timestamp").reset_index(drop=True)

# Extract time features
df_time["year"]         = df_time["timestamp"].dt.year
df_time["month"]        = df_time["timestamp"].dt.month
df_time["week"]         = df_time["timestamp"].dt.isocalendar().week.astype(int)
df_time["day_of_week"]  = df_time["timestamp"].dt.day_name()
df_time["hour"]         = df_time["timestamp"].dt.hour

# Shift block from hour: a more precise derivation than the noisy 'shift' column
def hour_to_shift(h):
    if 6 <= h < 14:
        return "Morning"
    elif 14 <= h < 22:
        return "Evening"
    else:
        return "Night"

df_time["shift_derived"] = df_time["hour"].apply(hour_to_shift)

df_time[["batch_id", "timestamp", "week", "day_of_week", "hour", "shift_derived"]].head(8)

,batch_id,timestamp,week,day_of_week,hour,shift_derived
0,B00002,2025-01-01 16:03:41,1,Wednesday,16,Evening
1,B00003,2025-01-01 19:16:57,1,Wednesday,19,Evening
2,B00004,2025-01-01 22:35:59,1,Wednesday,22,Night
3,B00005,2025-01-02 00:10:48,1,Thursday,0,Night
4,B00006,2025-01-02 02:33:13,1,Thursday,2,Night
5,B00007,2025-01-02 08:23:15,1,Thursday,8,Morning
6,B00008,2025-01-02 11:38:07,1,Thursday,11,Morning
7,B00009,2025-01-02 13:40:47,1,Thursday,13,Morning


### Rolling windows — detecting drift over time

A **rolling window** computes a statistic over a moving slice of rows.
For example, a 50-row rolling mean of scrap rate shows you whether quality is drifting up or down over recent production cycles.

![Rolling window explainer](rolling_window_explainer.svg)

```python
df["col"].rolling(window=50).mean()
```

The first 49 rows will be `NaN` (not enough data yet for a full window). `min_periods=1` removes this constraint.

In [30]:
# Calculate scrap rate first
df_time["scrap_rate"] = (df_time["defective_units"] / df_time["units_produced"]).round(4)

# Rolling average scrap rate — 50-cycle window
df_time["scrap_rate_roll50"] = df_time["scrap_rate"].rolling(window=50, min_periods=1).mean().round(4)

# Rolling std dev — useful for detecting instability / drift
df_time["scrap_rate_roll50_std"] = df_time["scrap_rate"].rolling(window=50, min_periods=5).std().round(4)

# Per-machine rolling average (sort by machine first, then compute within group)
df_time["scrap_rate_roll50_by_machine"] = (
    df_time
    .sort_values(["machine_id", "timestamp"])
    .groupby("machine_id")["scrap_rate"]
    .transform(lambda s: s.rolling(50, min_periods=1).mean())
    .round(4)
)

df_time[["batch_id", "machine_id", "timestamp", "scrap_rate", "scrap_rate_roll50", "scrap_rate_roll50_by_machine"]].head(10)

,batch_id,machine_id,timestamp,scrap_rate,scrap_rate_roll50,scrap_rate_roll50_by_machine
0,B00002,M05,2025-01-01 16:03:41,0.0487,0.0487,0.0487
1,B00003,M03,2025-01-01 19:16:57,0.0579,0.0533,0.0579
2,B00004,M01,2025-01-01 22:35:59,0.0746,0.0604,0.0746
3,B00005,M04,2025-01-02 00:10:48,0.0761,0.0643,0.0761
4,B00006,M01,2025-01-02 02:33:13,0.0393,0.0593,0.0570
5,B00007,M06,2025-01-02 08:23:15,0.0518,0.0581,0.0518
6,B00008,M05,2025-01-02 11:38:07,0.0356,0.0549,0.0422
7,B00009,M02,2025-01-02 13:40:47,0.0791,0.0579,0.0791
8,B00010,M02,2025-01-02 13:45:31,0.0206,0.0537,0.0499
9,B00011,M01,2025-01-02 15:39:47,0.0559,0.0540,0.0566


### Weekly scrap trend

With the `week` column we extracted earlier, we can now build a weekly summary in one line.
This is the kind of view a factory quality manager looks at every Monday morning.

In [ ]:
weekly_summary = (
    df_time
    .groupby(["year", "week"])
    .agg(
        batches=("batch_id", "count"),
        total_produced=("units_produced", "sum"),
        total_defective=("defective_units", "sum"),
        avg_scrap_rate=("scrap_rate", "mean"),
    )
    .assign(avg_scrap_rate=lambda d: d["avg_scrap_rate"].round(4))
    .reset_index()
)

weekly_summary.head(12)

---

## Part 9 — Method Chaining: clean pipelines

So far we have been cleaning the data step by step, mutating `df_clean` in place.
That is fine for learning, but in production code you want a **repeatable, readable pipeline**.

Pandas supports **method chaining** — each method returns a DataFrame, so you can chain `.method()` calls:

```python
result = (
    df
    .pipe(step_one)
    .pipe(step_two)
    .assign(new_col=lambda d: d["a"] / d["b"])
    .query("new_col > 0.05")
    .sort_values("new_col", ascending=False)
    .reset_index(drop=True)
)
```

`.pipe(fn)` is the key — it passes the DataFrame as the first argument to any function you write.
This lets you define named transformation steps and compose them cleanly.

Below is the full cleaning pipeline for `production_log`, refactored as a chain.

In [31]:
SHIFT_MAP = {
    "Morning": "Morning", "morning": "Morning", "AM":      "Morning",
    "Evening": "Evening", "EVENING": "Evening", "Eve":     "Evening",
    "Night":   "Night",   "night":   "Night",   "PM":      "Night", "N": "Night",
}


def fix_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["units_produced", "defective_units", "cycle_time_sec"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())
    return df


def fix_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    return df


def fill_missing_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    df["operator_id"] = df["operator_id"].fillna("UNKNOWN")
    df["material_batch_id"] = df["material_batch_id"].fillna("MISSING")
    return df


def fill_missing_sensors(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["temperature_c", "pressure_bar"]:
        medians = df.groupby("machine_id")[col].transform("median")
        df[col] = df[col].fillna(medians)
    return df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates(subset=["batch_id"], keep="first").reset_index(drop=True)


def normalise_shift(df: pd.DataFrame) -> pd.DataFrame:
    df["shift"] = df["shift"].map(SHIFT_MAP).fillna("UNKNOWN")
    return df


def cap_outliers(df: pd.DataFrame) -> pd.DataFrame:
    df["units_produced"] = df["units_produced"].clip(upper=800)
    return df


# ── The full pipeline as a single chain ──────────────────────────────────────
prod_pipeline = (
    pd.read_parquet("data/production_log.parquet")
    .pipe(fix_numeric_columns)
    .pipe(fix_timestamp)
    .pipe(fill_missing_categoricals)
    .pipe(fill_missing_sensors)
    .pipe(remove_duplicates)
    .pipe(normalise_shift)
    .pipe(cap_outliers)
    .assign(
        scrap_rate=lambda d: (d["defective_units"] / d["units_produced"]).round(4),
        week=lambda d: d["timestamp"].dt.isocalendar().week.astype("Int64"),
        shift_derived=lambda d: d["timestamp"].dt.hour.apply(hour_to_shift),
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(f"Final clean production table: {prod_pipeline.shape}")
prod_pipeline[["batch_id", "timestamp", "machine_id", "shift", "scrap_rate", "week"]].head(8)

Final clean production table: (12000, 15)


,batch_id,timestamp,machine_id,shift,scrap_rate,week
0,B00002,2025-01-01 16:03:41,M05,Evening,0.0487,1
1,B00003,2025-01-01 19:16:57,M03,Evening,0.0579,1
2,B00004,2025-01-01 22:35:59,M01,Evening,0.0746,1
3,B00005,2025-01-02 00:10:48,M04,Morning,0.0761,1
4,B00006,2025-01-02 02:33:13,M01,Morning,0.0393,1
5,B00007,2025-01-02 08:23:15,M06,Morning,0.0518,1
6,B00008,2025-01-02 11:38:07,M05,Morning,0.0356,1
7,B00009,2025-01-02 13:40:47,M02,Evening,0.0791,1


---

## Part 10 — Business questions on clean data

Now that the data is clean, we can finally ask real questions.
Notice how *none* of these questions would give a correct answer on the raw data.

This is why cleaning comes first.

In [32]:
# Q1: Which machine has the worst average scrap rate?
(
    prod_pipeline
    .groupby("machine_id")["scrap_rate"]
    .mean()
    .sort_values(ascending=False)
    .rename("avg_scrap_rate")
    .reset_index()
    .assign(avg_scrap_rate=lambda d: d["avg_scrap_rate"].round(4))
)

,machine_id,avg_scrap_rate
0,M02,0.0647
1,M04,0.0646
2,M06,0.0641
3,M03,0.0641
4,M01,0.0633
5,M05,0.0633


In [33]:
# Q2: Is scrap rate trending up or down over the weeks?
(
    prod_pipeline
    .dropna(subset=["week"])
    .groupby("week")["scrap_rate"]
    .mean()
    .round(4)
    .reset_index()
    .rename(columns={"scrap_rate": "avg_scrap_rate"})
    .head(15)
)

,week,avg_scrap_rate
0,1,0.0661
1,2,0.0634
2,3,0.0666
3,4,0.0611
4,5,0.0677
5,6,0.0667
6,7,0.0635
7,8,0.0614
8,9,0.0632
9,10,0.0621


In [34]:
# Q3: Do defect rates differ by shift?
(
    prod_pipeline
    .groupby("shift_derived")
    .agg(
        batches=("batch_id", "count"),
        avg_scrap_rate=("scrap_rate", "mean"),
        total_defective=("defective_units", "sum"),
    )
    .assign(avg_scrap_rate=lambda d: d["avg_scrap_rate"].round(4))
    .sort_values("avg_scrap_rate", ascending=False)
)

,batches,avg_scrap_rate,total_defective
shift_derived,,,
Morning,3810,0.0647,117420.0
Night,4389,0.0639,133043.0
Evening,3801,0.0634,113822.0


In [35]:
# Q4: Which week had the single worst day for any machine? (cross-tab: machine × week)
machine_week = (
    prod_pipeline
    .dropna(subset=["week"])
    .groupby(["machine_id", "week"])["scrap_rate"]
    .mean()
    .round(4)
    .reset_index()
    .pivot(index="week", columns="machine_id", values="scrap_rate")
)

machine_week.head(10)

machine_id,M01,M02,M03,M04,M05,M06
week,,,,,,
1,0.0721,0.0584,0.0642,0.0660,0.0722,0.0646
2,0.0658,0.0602,0.0630,0.0677,0.0616,0.0609
3,0.0671,0.0730,0.0593,0.0714,0.0654,0.0638
4,0.0564,0.0592,0.0621,0.0590,0.0650,0.0642
5,0.0673,0.0719,0.0685,0.0644,0.0602,0.0727
6,0.0679,0.0640,0.0698,0.0644,0.0680,0.0663
7,0.0611,0.0570,0.0643,0.0658,0.0625,0.0696
8,0.0689,0.0672,0.0590,0.0535,0.0555,0.0631
9,0.0676,0.0609,0.0609,0.0530,0.0663,0.0658


---

## Part 11 — Save the clean data back to Parquet

Once cleaned, save the result as a new Parquet file.
Never overwrite the raw data — always write a new file with a `_clean` suffix or into a `processed/` directory.
This preserves your ability to re-run the pipeline from scratch.

In [36]:
from pathlib import Path

processed_dir = Path("data/processed")
processed_dir.mkdir(exist_ok=True)

output_path = processed_dir / "production_log_clean.parquet"
prod_pipeline.to_parquet(output_path, index=False)

print(f"Clean table saved → {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"Rows: {len(prod_pipeline):,}")

Clean table saved → data/processed/production_log_clean.parquet
File size: 368.6 KB
Rows: 12,000


---

## Homework

Choose one of the three options. They increase in difficulty and reward.

---

### Option A — Clean and document

Apply the same cleaning pipeline to `inspection_log.parquet`:

1. Run `isna().sum()` and decide a strategy for each column
2. Fix the `defect_type` column — normalise all 18+ variants to 6 canonical categories
3. Fix the `severity` column — normalise `"HIGH"`, `"high"`, `"med"`, `"MEDIUM"`, etc.
4. Fix the `passed` column — it contains `True`, `False`, `"yes"`, `"no"`, `"1"`, `"0"`, and `NaN`. Convert to a clean boolean.
5. Write a Markdown summary cell explaining every decision you made and why

---

### Option B — Cross-table business questions

Using the three-table merge we built in Part 7:

1. Do defect rates differ by material supplier? (join `production_log` + `material_batches` + `inspection_log`)
2. Which supplier produced the most batches where `certificate_ok` is False?
3. Is there a relationship between `material_grade` and `scrap_rate`? Show it numerically.
4. Which machine + supplier combination has the worst average scrap rate?
5. Are `any_critical` inspection flags more common for a specific machine or shift?

Present each answer as a Pandas DataFrame with a 1–2 sentence business interpretation.

---

### Option C — Automated data quality report (hardest)

Write a Python function:

```python
def data_quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    ...
```

That returns a DataFrame with one row per column and these columns:

| column_name | dtype | missing_pct | n_unique | has_numeric_strings | has_mixed_date_formats | suspected_issue |
|---|---|---|---|---|---|---|

- `has_numeric_strings`: `True` if dtype is `object` but >50% of values can be coerced by `pd.to_numeric`
- `has_mixed_date_formats`: `True` if dtype is `object` and >30% of values can be coerced by `pd.to_datetime`
- `suspected_issue`: a human-readable string summarising the main problem (e.g. `"Numeric column stored as string - coerce recommended"`)

Run it on all three raw Parquet files and display the combined report.

---

## Session summary

| Topic | Key function(s) |
|---|---|
| Loading Parquet | `pd.read_parquet()` / `df.to_parquet()` |
| Missing values | `isna()`, `fillna()`, `dropna()`, `groupby().transform()` |
| Type coercion | `pd.to_numeric(errors='coerce')`, `pd.to_datetime(errors='coerce')` |
| Duplicates | `duplicated()`, `drop_duplicates(subset=...)` |
| Category normalisation | `value_counts()`, `.map()`, `.replace()` |
| Merging | `df.merge(other, on=..., how='left')` |
| Datetime features | `.dt.hour`, `.dt.isocalendar().week`, `.dt.day_name()` |
| Rolling windows | `.rolling(window=N).mean()` / `.std()` |
| Chaining | `.pipe()`, `.assign()`, `.query()` |

---

> **Remember:** cleaning decisions are **business decisions**. Every `fillna()` and every `drop_duplicates()` call makes an assumption about what the data means. Document every assumption. If the cleaning logic is wrong, every analysis built on top of it is wrong too.